# Davinci Notebook 13 — Agent & Tool-Use Training Data

Build a validated JSONL schema for agent trajectories: planning (optional), tool selection, JSON arguments, observations, final response, and error recovery. It does not synthesize verified trajectories; validate examples and tool outcomes independently.

In [ ]:
import json, hashlib, re
from pathlib import Path


## 1. Canonical trajectory schema
Example format includes explicit tool result provenance and avoids treating observation text as model instructions.

In [ ]:
example={'trajectory_id':'example-001','language':'fa','messages':[{'role':'user','content':'...'}, {'role':'assistant','type':'tool_call','name':'search','arguments':{'query':'...'}},{'role':'tool','name':'search','content':'...','trusted':False},{'role':'assistant','content':'...'}], 'metadata':{'source':'synthetic','verified':False}}
def validate_trajectory(t):
 issues=[]
 if not isinstance(t,dict) or not isinstance(t.get('messages'),list): return ['trajectory must be object with messages list']
 for i,m in enumerate(t['messages']):
  if not isinstance(m,dict): issues.append(f'message {i}: not object'); continue
  role=m.get('role')
  if role not in ('user','assistant','tool','system'): issues.append(f'message {i}: invalid role')
  if m.get('type')=='tool_call':
   if role!='assistant': issues.append(f'message {i}: tool_call should be assistant-authored')
   if not isinstance(m.get('name'),str) or not m.get('name'): issues.append(f'message {i}: tool name missing')
   if not isinstance(m.get('arguments'),dict): issues.append(f'message {i}: arguments should be object')
  if role=='tool' and 'name' not in m: issues.append(f'message {i}: tool result missing name')
 return issues
print('example validation:',validate_trajectory(example))


## 2. Dataset quality and duplicate detection
Use a stable hash to detect exact duplicates; this is not semantic deduplication.

In [ ]:
def trajectory_hash(t): return hashlib.sha256(json.dumps(t,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def audit_trajectories(items):
 hashes=set(); report={'total':len(items),'invalid':0,'duplicates':0,'issues':[]}
 for i,t in enumerate(items):
  issues=validate_trajectory(t)
  if issues: report['invalid']+=1; report['issues'].append({'index':i,'issues':issues})
  h=trajectory_hash(t)
  if h in hashes: report['duplicates']+=1
  hashes.add(h)
 return report


## 3. Export JSONL and dataset card fields

In [ ]:
def export_jsonl(items,path):
 valid=[x for x in items if not validate_trajectory(x)]
 Path(path).write_text(''.join(json.dumps(x,ensure_ascii=False)+'\\n' for x in valid),encoding='utf-8'); return len(valid)
required_card_fields=['sources','collection_method','license','verification_method','tool_environment','privacy_review','known_limitations','version']
print('Dataset card fields:',required_card_fields)


## 4. Trajectory types to collect
- Tool choice and tool argument generation.
- Multi-tool sequencing and stopping decisions.
- Error handling and retry policies.
- Replanning after failed/partial observations.
- Tool-free tasks (recognizing when tools are unnecessary).
- Persian-language tools and bilingual interactions.
- MCP tools with permission boundaries and untrusted outputs.
- Browser, code, file, database, calculator, and retrieval tasks.
- Long task history with relevant-memory retrieval.

Use a mix of human-authored, verified programmatic, and carefully audited synthetic examples; record provenance and license.